# nb_pipeline_03_acl_reconcile — re-stamp `allowed_groups`/`allowed_users` on ACL drift

Fast path that keeps Azure AI Search security trimming in sync with `acls.json` **without**
re-running Document Intelligence or embeddings. For every already-indexed file whose resolved
ACL version differs from what's recorded in `ingestion_state`, it merge-patches only the
`allowed_groups` and `allowed_users` fields on that file's existing chunks and updates the stored `acl_version`.

Run this after editing `acls.json`. See `PRODUCT_SPEC.md` section 9 (ACL drift reconciliation).

Uses the **Azure AI Search REST API** via `requests` (always present in Fabric) — **no
`%pip install`**, since inline pip is rejected by the Fabric job runtime.

## Required permissions (identity running this notebook)
Auth is **hybrid** (a Fabric constraint). This notebook only talks to AI Search:

| Resource | Auth in Fabric |
| --- | --- |
| Azure AI Search | **Admin API key** from Key Vault (`kv_name`/`search_key_secret`) |

The running user needs Key Vault **secret get**. The Fabric lakehouse must be attached so
`spark.table('config')` and the ACL file resolve.


## Config + ACL map


In [ ]:
import json, hashlib, requests
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

cfg = {r['key']: r['value'] for r in spark.table('config').collect()}

acls_path = cfg.get('acls_file_path', 'Files/acls/acls.json')
raw = spark.read.text(acls_path, wholetext=True).collect()[0][0]
ACL_MAP = {f['path'].rstrip('/'): (f.get('groups', []), f.get('users', []))
           for f in json.loads(raw).get('folders', [])}

def _acl_version(groups, users):
    base = '|'.join(groups)
    if users:
        base += '#' + '|'.join(users)
    return hashlib.sha256(base.encode()).hexdigest()[:16]

def resolve_acl(rel_path):
    parts = rel_path.split('/')
    for i in range(len(parts) - 1, 0, -1):
        prefix = '/'.join(parts[:i])
        if prefix in ACL_MAP:
            groups = sorted(ACL_MAP[prefix][0])
            users = sorted(ACL_MAP[prefix][1])
            return groups, users, _acl_version(groups, users)
    return [], [], None

def to_rel(file_path):
    m = '/Files/'
    return 'Files/' + file_path.split(m, 1)[1] if m in file_path else file_path


## Auth — AI Search admin key from Key Vault (REST)
Fabric can't mint an AI Search token, so we read the **admin API key from Key Vault**
(`kv_name`/`search_key_secret`) and send it in the `api-key` header. The running user needs Key
Vault **secret get**.


In [ ]:
import notebookutils

VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
SEARCH_KEY = notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret'])
SEARCH_ENDPOINT = cfg['search_endpoint'].rstrip('/')
INDEX_NAME = cfg['search_index_name']
SEARCH_API = '2024-07-01'
def _search_headers():
    return {'api-key': SEARCH_KEY, 'Content-Type': 'application/json'}

def restamp(file_path, groups, users):
    """Merge-patch allowed_groups + allowed_users on all chunks for the file. Returns chunk count."""
    search_url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={SEARCH_API}'
    index_url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/index?api-version={SEARCH_API}'
    safe = file_path.replace("'", "''")
    ids, skip = [], 0
    while True:
        body = {'search': '*', 'filter': f"file_path eq '{safe}'",
                'select': 'chunk_id', 'top': 1000, 'skip': skip}
        r = requests.post(search_url, headers=_search_headers(), json=body)
        r.raise_for_status()
        page = r.json().get('value', [])
        ids += [d['chunk_id'] for d in page]
        if len(page) < 1000:
            break
        skip += 1000
    if ids:
        actions = [{'@search.action': 'merge', 'chunk_id': i,
                    'allowed_groups': groups, 'allowed_users': users} for i in ids]
        for i in range(0, len(actions), 500):
            r = requests.post(index_url, headers=_search_headers(), json={'value': actions[i:i + 500]})
            r.raise_for_status()
    return len(ids)


## Detect drift and reconcile
Only files that are `complete` and already have index state are considered. A file drifts when
its currently-resolved `acl_version` differs from the one stored in `ingestion_state`.


In [ ]:
state = {r['file_path']: r['acl_version'] for r in
         spark.table('ingestion_state').select('file_path', 'acl_version').collect()}
complete = [r['file_path'] for r in spark.table('file_metadata')
            .where(F.col('process_status') == 'complete').select('file_path').collect()]

reconciled, patched_chunks = 0, 0
now = datetime.now(timezone.utc)
st = DeltaTable.forName(spark, 'ingestion_state')
for fp in complete:
    if fp not in state:
        continue
    groups, users, acl_version = resolve_acl(to_rel(fp))
    if acl_version == state[fp]:
        continue  # no drift
    n = restamp(fp, groups, users)
    st.update(F.col('file_path') == F.lit(fp),
              {'acl_version': F.lit(acl_version), 'indexed_utc': F.lit(now)})
    reconciled += 1; patched_chunks += n
    print(f'reconciled {fp}: {n} chunks -> {len(groups)} groups, {len(users)} users')

print(f'done: {reconciled} files re-stamped, {patched_chunks} chunks patched')
